# Full-video baseline and targeted review comparison

This streamlined notebook checks out the tested pipeline from GitHub instead of embedding its Python source. Attach the private dataset containing `video.mp4` and the target reference arrays, enable Internet and a GPU, configure the `HF_TOKEN` Kaggle secret, and run from the top.

The checkout is pinned to an exact commit for reproducibility. Videos, biometric references, tokens, checkpoints, and transcripts remain outside GitHub.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile

# Local: set this to the folder containing video.mp4 and the target .npy files.
# Kaggle: leave None; the attached dataset is found automatically.
DATA_DIR = None
RUN_FULL_VIDEO = True
RUN_GAP_RECOVERY = False  # Older separate text-only pass; targeted review already includes gaps
RUN_TARGETED_REVIEW = True
RUN_HEIGHT_CONTROL = False  # Opening sanity check is enough before the unchanged full baseline
REVIEW_WEAK_CONFIDENCE = 0.35
REVIEW_SHORT_SECONDS = 1.0
# External evaluation controls only; these are not rules inside the review algorithm.
REVIEW_CONTROL_REGIONS = [(640, 665), (665, 690), (1085, 1150)]
BATCH_SIZE = 4
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    # Fail before lengthy installation if this session has no GPU.
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('No GPU in this session. Enable GPU T4 x2 in Settings, then rerun this cell.')
    print(probe.stdout)
    matches = list(Path('/kaggle/input').rglob('video.mp4'))
    if DATA_DIR is None:
        if len(matches) != 1:
            raise RuntimeError('Attach the private video dataset; expected one video.mp4, or set DATA_DIR explicitly.')
        DATA_DIR = matches[0].parent
    BASE = Path('/kaggle/working')
else:
    # Convenience for the current local project; otherwise use the chosen folder.
    local_project = Path('/home/think/projects/whisperx_diarization')
    DATA_DIR = DATA_DIR or (local_project if local_project.exists() else Path.cwd())
    BASE = Path.cwd()/'diarization-run'
DATA = Path(DATA_DIR).expanduser().resolve()
for name in ('video.mp4',):
    if not (DATA/name).is_file():
        raise RuntimeError(f'Missing {name}. Set DATA_DIR to the project or extracted dataset folder.')
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'
WORK.mkdir(exist_ok=True)
RESULTS = BASE/'results'
RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
print('Input folder:', DATA)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)
REFERENCE = WORK/'target-reference'
REFERENCE.mkdir(exist_ok=True)


## Install and verify the environment
No activation or separate `wrapt`/ONNX repair cells are needed. Setup checks whether pip works, rather than only checking that Python exists. It uses virtualenv's bundled pip when creation or repair is needed. Installing dependencies can take several minutes; progress appears below.

In [ ]:
# Fetch the exact reviewed source revision. WORK already contains the empty
# target-reference directory created above, so initialize it in place.
REPO_URL = 'https://github.com/sleverbor/whisperx-diarization.git'
REPO_COMMIT = '265c47c3f6f232766ea3d7daacc78c906668f2f6'
subprocess.run(['git', '-C', str(WORK), 'init'], check=True)
subprocess.run(['git', '-C', str(WORK), 'remote', 'remove', 'origin'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['git', '-C', str(WORK), 'remote', 'add', 'origin', REPO_URL], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--depth', '1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
print('Pipeline source:', REPO_URL, REPO_COMMIT)

# Strip notebook-only display settings from every child process.
ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib')
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
ENV['PYTHONUNBUFFERED'] = '1'
ENV.pop('PYTHONPATH', None)

def checked(args, **kwargs):
    return subprocess.run(args, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    # Install bootstrap tools outside the runtime, without replacing notebook packages.
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy()
    bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)
checked([PYTHON, '-m', 'pip', '--version'])
print('2/4: Installing pipeline dependencies (including wrapt in the venv)', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt']
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
# InsightFace's metadata requires the CPU-named distribution, even though GPU
# supplies the same import. Check dependency resolution before removing that overlap.
print('3/4: Removing overlapping ONNX packages and installing CUDA 12 build', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('Install ffmpeg on this laptop and rerun setup. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')
print('4/4: Verifying imports, GPU and behavior checks', flush=True)
verification = """import torch,onnxruntime as ort,wrapt
import chainofrules
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX:', ort.__version__, ort.__file__)
print('Advertised providers:', ort.get_available_providers())
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is not available to this runtime'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX package missing'"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers'], cwd=WORK)
print('Setup complete. Continue to credentials and tests.', flush=True)

import hashlib
reference_hashes = {}
for name in ('voice_embeddings.npy', 'face_embeddings.npy', 'opening_officer_reference.npy'):
    direct = DATA/name
    matches = [direct] if direct.is_file() else list(DATA.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one private {name} in the attached dataset; found {len(matches)}')
    content = matches[0].read_bytes()
    (REFERENCE/name).write_bytes(content)
    reference_hashes[name] = hashlib.sha256(content).hexdigest()
(RESULTS/'reference-hashes.json').write_text(json.dumps(reference_hashes, indent=2))
checked([PYTHON, '-m', 'unittest', 'test_transcript_gaps', 'test_window_review', 'test_review_regions'], cwd=WORK)
print('Private references ready:', reference_hashes)


## Credentials and optional checkpoint restore
The token is never embedded in this notebook or printed. Kaggle reads the authorized secret; on a laptop, use the environment variable or the hidden prompt. If restoring checkpoints, attach the downloaded archive as another private dataset. Completed stages with an exact fingerprint are reused; interrupted stages rerun.

In [ ]:
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    ENV['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A token with diarization-model access is required.')
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
print('Credentials configured; token not displayed.')

In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', CACHE.parent, CACHE.name)

def run_test(video_name, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(Path(video_name) if Path(video_name).is_absolute() else DATA/video_name),
               '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
               '--face-priors', str(REFERENCE/'face_embeddings.npy'),
               '--output', str(output), '--cache-dir', str(CACHE),
               '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        with (RESULTS/(stem+'.log')).open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                # Never save or display the secret even if a dependency prints it.
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush()
                print(line if len(line) < 1000 else line[:1000]+' ... [full line saved in log]\n', end='')
            if process.wait() != 0:
                raise RuntimeError('Test failed; see the log. For CUDA out-of-memory, retry with batch_size=1.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    with (RESULTS/'runtime-packages.txt').open('w') as packages:
        checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Runtime:', result.get('runtime'))
    return result

def extract_clip(start, duration, name):
    clip = WORK/name
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-ss', str(start), '-i', str(DATA/'video.mp4'), '-t', str(duration),
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-c:a', 'aac', str(clip)])
    return clip


def run_targeted_review():
    output_dir = RESULTS/'targeted-review'
    command = [PYTHON, str(WORK/'review_transcript_regions.py'),
        '--video', str(DATA/'video.mp4'),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--target-reference', str(REFERENCE/'voice_embeddings.npy'),
        '--other-reference', 'Opening_officer='+str(REFERENCE/'opening_officer_reference.npy'),
        '--output-dir', str(output_dir), '--cache-dir', str(CACHE/'targeted-review'),
        '--weak-confidence', str(REVIEW_WEAK_CONFIDENCE),
        '--short-seconds', str(REVIEW_SHORT_SECONDS),
        '--minimum-gap', '5', '--context-seconds', '3',
        '--maximum-window-seconds', '30', '--window-overlap-seconds', '4',
        '--device', 'cuda' if ON_KAGGLE else 'auto']
    if not ON_KAGGLE:
        command += ['--speechbrain-cache', str(DATA/'pretrained_models/spkrec-ecapa-voxceleb')]
    for start, end in REVIEW_CONTROL_REGIONS:
        command += ['--extra-region', f'{start}:{end}']
    baseline_before = (RESULTS/'full_video_evidence.json').read_bytes()
    started = time.monotonic()
    try:
        with (RESULTS/'targeted-review.log').open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush(); print(line, end='')
            if process.wait() != 0:
                raise RuntimeError('Targeted review failed; see targeted-review.log. Checkpoints were retained.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == baseline_before
    summary = json.loads((output_dir/'comparison_summary.json').read_text())
    print(f"Targeted review elapsed: {(time.monotonic()-started)/60:.1f} minutes")
    print(json.dumps(summary, indent=2))
    return summary


## Known-clip checks, baseline, then supplemental review

The 30-second opening clip validates the unchanged baseline and actual GPU providers first. Set `RUN_HEIGHT_CONTROL=True` only if you also want to repeat the previously validated 65-second height/weight control. The full video then runs once with the current pipeline. The review pass selects regions from that result and uses the same target voice reference plus an independently captured opening-speaker comparison reference. The name `Opening_officer` describes the supplied reference provenance; it is not police recognition.

Selected windows and their total duration are printed before review decoding. Every window is checkpointed. If Kaggle stops, download `stage-checkpoints.zip`, attach it with the video on the next session, and rerun the notebook. Duplicate hypotheses from overlapping windows remain visible for comparison rather than being silently merged.


In [ ]:
opening_clip = extract_clip(0, 30, 'opening_30s.mp4')
controls = [(opening_clip, 'opening', 0)]
if RUN_HEIGHT_CONTROL:
    second_clip = extract_clip(625, 65, 'height_weight_65s.mp4')
    controls.append((second_clip, 'height_weight', 625))
for clip, stem, offset in controls:
    result = run_test(str(clip), stem, BATCH_SIZE)
    (RESULTS/(stem+'_source_offset.json')).write_text(json.dumps({'source_offset_seconds':offset}))
    providers = result.get('runtime', {}).get('face_providers', {})
    if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
        raise RuntimeError('Actual face-model GPU check failed. Full video has not started; review the log.')
    print((RESULTS/(stem+'_transcript.txt')).read_text())
if RUN_FULL_VIDEO:
    full_video = run_test('video.mp4', 'full_video', BATCH_SIZE)
    if RUN_GAP_RECOVERY:
        recovery_dir = RESULTS/('gap-review-' + str(time.time_ns()))
        try:
            checked([PYTHON, str(WORK/'recover_transcript_gaps.py'), str(DATA/'video.mp4'),
                     '--baseline', str(RESULTS/'full_video_evidence.json'),
                     '--output-dir', str(recovery_dir), '--minimum-gap', '5',
                     '--window-seconds', '20', '--overlap-seconds', '10', '--context-seconds', '2'], cwd=WORK)
        finally:
            export_checkpoints()
        recovered = json.loads((recovery_dir/'transcript_with_candidates.json').read_text())
        assert recovered['segments'] == full_video['segments']
        print('Gap review saved separately:', recovery_dir)
    if RUN_TARGETED_REVIEW:
        targeted_review = run_targeted_review()
else:
    print('Full video disabled. Review the known-clip results, then set RUN_FULL_VIDEO=True.')


## Save results

Download both archives. `diarization-results.zip` contains the unchanged baseline, gap candidates, targeted review hypotheses, readable transcripts, logs, selection details, and the comparison summary. `stage-checkpoints.zip` permits exact review-window reuse after a Kaggle interruption.


In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS)
display(FileLink(str(BASE/'diarization-results.zip')))
display(FileLink(str(BASE/'stage-checkpoints.zip')))
print('Saved in:', BASE)